# Настройка YAML конфигурации модели

Читает и обновляет `companies/{company}/configs/project.yaml`.

**Использование:** задайте компанию → запустите все ячейки → измените параметры → нажмите **Сохранить**

In [ ]:
import sys, yaml
from pathlib import Path
from IPython.display import display
import ipywidgets as w

NOTEBOOK_DIR = Path().resolve()
ROOT = NOTEBOOK_DIR.parents[2]   if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COMPANY_ID = 'us_steel'
CONFIG_PATH = ROOT / 'companies' / COMPANY_ID / 'configs' / 'project.yaml'

def load_yaml():
    if CONFIG_PATH.exists():
        with open(CONFIG_PATH) as f:
            return yaml.safe_load(f) or {}
    return {}

def save_yaml(cfg):
    with open(CONFIG_PATH, 'w') as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False, default_flow_style=False)
    print(f'Сохранено: {CONFIG_PATH}')

def get(cfg, *keys, default=None):
    v = cfg
    for k in keys:
        if not isinstance(v, dict) or k not in v:
            return default
        v = v[k]
    return v

def set_nested(cfg, keys, value):
    d = cfg
    for k in keys[:-1]:
        d = d.setdefault(k, {})
    d[keys[-1]] = value

cfg = load_yaml()
print(f'Загружен: {CONFIG_PATH.name}')
print(f'Компания: {get(cfg, "company", "name", default=COMPANY_ID)}')

## Горизонт прогноза

In [ ]:
mode = get(cfg, 'model', 'mode', default='standard')
mode_cfg = get(cfg, 'model', mode, default=get(cfg, 'model', 'standard', default={}))
periods = get(mode_cfg, 'periods', default={})

w_hist_start = w.IntText(value=get(periods, 'history_start_year', default=2010), description='История от:')
w_hist_end   = w.IntText(value=get(periods, 'history_end_year',   default=2024), description='История до:')
w_fc_start   = w.IntText(value=get(periods, 'forecast_start_year',default=2025), description='Прогноз от:')
w_fc_end     = w.IntText(value=get(periods, 'forecast_end_year',  default=2029), description='Прогноз до:')

display(w.HBox([w_hist_start, w_hist_end, w_fc_start, w_fc_end]))

## Revenue & Margins

In [ ]:
margins = get(mode_cfg, 'margins', default={})

w_cogs_pct = w.FloatText(value=get(margins, 'cogs_pct', default=0.0),
    description='COGS %:', style={'description_width':'120px'})
w_cogs_note = w.Label('(0 = из препроцессора)')

w_tax_rate = w.FloatText(value=get(margins, 'tax_rate_statutory', default=0.21),
    description='Tax rate:', style={'description_width':'120px'})

display(w.VBox([
    w.HBox([w_cogs_pct, w_cogs_note]),
    w.HBox([w_tax_rate, w.Label('(statutory rate, override EWA)')]),
]))

## Working Capital (дни)

In [ ]:
wc = get(mode_cfg, 'wc', default={})

w_dso = w.FloatText(value=get(wc,'dso_days',default=0), description='DSO days:', style={'description_width':'120px'})
w_dih = w.FloatText(value=get(wc,'dio_days',default=0), description='DIH days:', style={'description_width':'120px'})
w_dpo = w.FloatText(value=get(wc,'dpo_days',default=0), description='DPO days:', style={'description_width':'120px'})
w_wc_note = w.Label('(0 = из препроцессора)')

display(w.VBox([
    w.HBox([w_dso, w_dih, w_dpo]),
    w_wc_note,
]))

## CapEx & PP&E

In [ ]:
capex = get(mode_cfg, 'capex_policy', default={})
ppe   = get(mode_cfg, 'ppe', default={})

w_capex_pct = w.FloatText(value=get(capex,'ratio_default',default=0.0),
    description='CapEx/Rev %:', style={'description_width':'120px'})
w_dep_rate  = w.FloatText(value=get(ppe,'depreciation_rate',default=0.0),
    description='Dep rate:', style={'description_width':'120px'})
w_capex_note = w.Label('(0 = из препроцессора)')

display(w.VBox([
    w.HBox([w_capex_pct, w_dep_rate]),
    w_capex_note,
]))

## Долг

In [ ]:
debt = get(mode_cfg, 'debt', default={})
rc   = get(debt, 'rc', default={})

w_debt_mode = w.Dropdown(
    options=['auto','schedule_based','optimizer','parametric'],
    value=get(debt,'mode',default='auto'),
    description='Debt mode:', style={'description_width':'120px'})
w_debt_target = w.FloatText(value=get(debt,'target_pct_revenue',default=0.35),
    description='Target debt/rev:', style={'description_width':'120px'})
w_avg_rate    = w.FloatText(value=get(debt,'avg_rate_pct',default=0.05),
    description='Avg rate:', style={'description_width':'120px'})
w_rc_enable   = w.Checkbox(value=get(rc,'enable',default=True), description='RC enabled')
w_rc_limit    = w.FloatText(value=get(rc,'limit',default=0.0),
    description='RC limit:', style={'description_width':'120px'})
w_min_cash    = w.FloatText(value=get(rc,'min_cash',default=0.0),
    description='Min cash:', style={'description_width':'120px'})
w_refi_mode   = w.Dropdown(
    options=['simple','new','auto'],
    value=get(debt,'refinancing',{}).get('mode','simple'),
    description='Refi mode:', style={'description_width':'120px'})

display(w.VBox([
    w.HBox([w_debt_mode, w_debt_target, w_avg_rate]),
    w.HBox([w_rc_enable, w_rc_limit, w_min_cash]),
    w.HBox([w_refi_mode]),
]))

## Лизинг

In [ ]:
leases = get(mode_cfg, 'leases', default={})
lease_cork = get(mode_cfg, 'lease', default={})
taxes_cfg  = get(mode_cfg, 'taxes', default={})

# Lease model settings
w_lease_enabled = w.Checkbox(value=get(leases,'enabled',default=False), description='Lease enabled')
w_lease_std     = w.Dropdown(
    options=['US_GAAP','IFRS'],
    value=get(cfg,'company','accounting_standard',default='US_GAAP'),
    description='Standard:', style={'description_width':'120px'})
w_lease_rate    = w.FloatText(value=get(leases,'default_discount_rate',default=0.05),
    description='Discount rate:', style={'description_width':'120px'})

# Operating lease corkscrew params (filled from preprocessor by default)
w_op_decay_rate  = w.FloatText(value=get(lease_cork,'op_decay_rate',default=0.33),
    description='Op decay rate:', style={'description_width':'130px'})
w_op_new_leases  = w.FloatText(value=get(lease_cork,'op_new_leases',default=0.0),
    description='Op new ($M):', style={'description_width':'130px'})
w_op_cash_pay    = w.FloatText(value=get(lease_cork,'op_cash_payment',default=0.0),
    description='Op cash pay ($M):', style={'description_width':'130px'})

# NOL carryforward (TCJA rules)
w_nol_balance = w.FloatText(value=get(taxes_cfg,'nol_opening_balance',default=0.0),
    description='NOL open ($M):', style={'description_width':'130px'})
w_nol_limit   = w.FloatText(value=get(taxes_cfg,'nol_max_utilization_pct',default=0.80),
    description='NOL limit pct:', style={'description_width':'130px'})

display(w.HBox([w_lease_enabled, w_lease_std, w_lease_rate]))
display(w.HTML('<b>Operating Lease Corkscrew:</b>'))
display(w.HBox([w_op_decay_rate, w_op_new_leases, w_op_cash_pay]))
display(w.HTML('<b>NOL (TCJA — 80% of taxable income):</b>'))
display(w.HBox([w_nol_balance, w_nol_limit]))


## Equity

In [ ]:
equity = get(mode_cfg, 'equity', default={})
feats  = get(cfg, 'features', default={})

w_div_payout    = w.FloatText(value=get(equity,'dividend_payout_ratio',default=0.0),
    description='Div payout:', style={'description_width':'130px'})
w_buyback_pct   = w.FloatText(value=get(equity,'buyback_pct_fcf',default=0.0),
    description='Buyback%FCF:', style={'description_width':'130px'})
w_buyback_lev   = w.FloatText(value=get(equity,'buyback_leverage_max',default=2.0),
    description='Buyback lev max:', style={'description_width':'130px'})
w_min_cash_f    = w.FloatText(value=get(feats,'min_cash',default=0.0),
    description='Min cash $:', style={'description_width':'130px'})

display(w.HBox([w_div_payout, w_buyback_pct, w_buyback_lev, w_min_cash_f]))


## Features (вкл/выкл модулей)

In [ ]:
w_use_ppe_cork   = w.Checkbox(value=get(feats,'use_ppe_corkscrew',default=True),    description='PPE corkscrew')
w_use_wc_days    = w.Checkbox(value=get(feats,'use_wc_days',default=True),          description='WC days')
w_use_debt_rc    = w.Checkbox(value=get(feats,'use_debt_rc',default=True),          description='Debt RC')
w_use_intang     = w.Checkbox(value=get(feats,'use_intangibles_corkscrew',default=True), description='Intangibles cork')
w_use_tax        = w.Checkbox(value=get(feats,'use_tax_corkscrew',default=True),    description='Tax corkscrew')
w_use_int_pay    = w.Checkbox(value=get(feats,'use_interest_payable_cork',default=True), description='Interest payable')

display(w.VBox([
    w.HBox([w_use_ppe_cork, w_use_wc_days, w_use_debt_rc]),
    w.HBox([w_use_intang, w_use_tax, w_use_int_pay]),
]))

## Сохранить конфигурацию

In [ ]:
def collect_and_save(_=None):
    cfg2 = load_yaml()
    mode2 = get(cfg2,'model','mode',default='standard')
    
    def s(*keys, val):
        set_nested(cfg2, list(keys), val)
    
    # Periods
    s('model',mode2,'periods','history_start_year', val=w_hist_start.value)
    s('model',mode2,'periods','history_end_year',   val=w_hist_end.value)
    s('model',mode2,'periods','forecast_start_year',val=w_fc_start.value)
    s('model',mode2,'periods','forecast_end_year',  val=w_fc_end.value)
    
    # Margins
    if w_cogs_pct.value > 0:
        s('model',mode2,'margins','cogs_pct', val=round(w_cogs_pct.value, 4))
    s('model',mode2,'margins','tax_rate_statutory', val=round(w_tax_rate.value, 4))
    
    # WC
    if w_dso.value > 0: s('model',mode2,'wc','dso_days', val=w_dso.value)
    if w_dih.value > 0: s('model',mode2,'wc','dio_days', val=w_dih.value)
    if w_dpo.value > 0: s('model',mode2,'wc','dpo_days', val=w_dpo.value)
    
    # CapEx
    if w_capex_pct.value > 0:
        s('model',mode2,'capex_policy','ratio_default', val=round(w_capex_pct.value, 4))
    if w_dep_rate.value > 0:
        s('model',mode2,'ppe','depreciation_rate', val=round(w_dep_rate.value, 4))
    
    # Debt
    s('model',mode2,'debt','mode',              val=w_debt_mode.value)
    s('model',mode2,'debt','target_pct_revenue',val=round(w_debt_target.value, 4))
    s('model',mode2,'debt','avg_rate_pct',      val=round(w_avg_rate.value, 4))
    s('model',mode2,'debt','rc','enable',       val=w_rc_enable.value)
    s('model',mode2,'debt','rc','limit',        val=w_rc_limit.value)
    s('model',mode2,'debt','rc','min_cash',     val=w_min_cash.value)
    s('model',mode2,'debt','refinancing','mode',val=w_refi_mode.value)
    
    # Leases
    s('model',mode2,'leases','enabled',                val=w_lease_enabled.value)
    s('model',mode2,'leases','default_discount_rate',  val=round(w_lease_rate.value,4))
    s('company','accounting_standard',                 val=w_lease_std.value)
    
    # Equity
    s('model',mode2,'equity','dividend_payout_ratio', val=round(w_div_payout.value,4))
    s('features','min_cash', val=w_min_cash_f.value)
    
    # Features
    s('features','use_ppe_corkscrew',        val=w_use_ppe_cork.value)
    s('features','use_wc_days',              val=w_use_wc_days.value)
    s('features','use_debt_rc',              val=w_use_debt_rc.value)
    s('features','use_intangibles_corkscrew',val=w_use_intang.value)
    s('features','use_tax_corkscrew',        val=w_use_tax.value)
    s('features','use_interest_payable_cork',val=w_use_int_pay.value)
    

    # NOL config
    s('model', mode2, 'taxes', 'nol_opening_balance', val=round(w_nol_balance.value, 3))
    s('model', mode2, 'taxes', 'nol_max_utilization_pct', val=round(w_nol_limit.value, 4))

    # Lease corkscrew params
    s('model', mode2, 'lease', 'op_decay_rate',   val=round(w_op_decay_rate.value, 4))
    if w_op_new_leases.value > 0:
        s('model', mode2, 'lease', 'op_new_leases',  val=round(w_op_new_leases.value, 2))
    if w_op_cash_pay.value > 0:
        s('model', mode2, 'lease', 'op_cash_payment', val=round(w_op_cash_pay.value, 2))

    # Equity
    s('model', mode2, 'equity', 'buyback_pct_fcf', val=round(w_buyback_pct.value, 4))
    s('model', mode2, 'equity', 'buyback_leverage_max', val=round(w_buyback_lev.value, 2))

    save_yaml(cfg2)
    print('✅ Конфигурация сохранена')
    print(f'   Прогноз: {w_fc_start.value}–{w_fc_end.value}')
    print(f'   Debt mode: {w_debt_mode.value}')
    print(f'   Tax rate: {w_tax_rate.value:.1%}')

btn_save = w.Button(
    description='💾 Сохранить',
    button_style='success',
    layout=w.Layout(width='200px', height='40px')
)
btn_save.on_click(collect_and_save)

btn_reload = w.Button(
    description='🔄 Перечитать YAML',
    button_style='info',
    layout=w.Layout(width='200px', height='40px')
)
def reload_yaml(_):
    global cfg
    cfg = load_yaml()
    print('Перезагружен:', CONFIG_PATH.name)
btn_reload.on_click(reload_yaml)

display(w.HBox([btn_save, btn_reload]))

## Текущее состояние YAML

In [ ]:
import yaml as _yaml
current = load_yaml()
# Показываем только ключевые секции
for section in ['company', 'features']:
    if section in current:
        print(f'--- {section} ---')
        print(_yaml.dump(current[section], allow_unicode=True, default_flow_style=False))

mode3 = current.get('model', {}).get('mode', 'standard')
mode_data = current.get('model', {}).get(mode3, {})
for section in ['periods', 'margins', 'wc', 'debt', 'leases', 'equity']:
    if section in mode_data:
        print(f'--- model.{mode3}.{section} ---')
        print(_yaml.dump(mode_data[section], allow_unicode=True, default_flow_style=False))